In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/"
SAVE_DIR = os.path.join(BASE_DIR, "attention_fusion_results_final")

os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving to:", SAVE_DIR)

Saving to: /content/drive/MyDrive/attention_fusion_results_final


In [ ]:
train_df = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_train.csv")
val_df   = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_val.csv")
test_df  = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_test.csv")

features = ["DR_prob", "Gl_prob", "AMD_prob", "DED_prob",
            "is_fundus", "is_oct", "is_slitlamp"]

targets = ["DR", "Glaucoma", "AMD", "DED"]

X_train = train_df[features].values
y_train = train_df[targets].values

X_val = val_df[features].values
y_val = val_df[targets].values

X_test = test_df[features].values
y_test = test_df[targets].values

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(FusionDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(FusionDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader  = DataLoader(FusionDataset(X_test, y_test), batch_size=32, shuffle=False)

In [ ]:
class AttentionFusion(nn.Module):
    def __init__(self, input_dim=7):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, input_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        attn = self.attention(x)
        weighted = x * (1 + attn)
        out = self.classifier(weighted)
        return out

In [ ]:
model = AttentionFusion().to(device)

pos_weights = torch.tensor([2.6, 1.5, 1.0, 1.0]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
epochs = 25
best_val_f1 = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ===== VALIDATION =====
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            out = model(xb)
            probs = torch.sigmoid(out / 1.2)

            val_preds.append(probs.cpu().numpy())
            val_targets.append(yb.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)

    # 🔥 better metric than raw 0.5 threshold
    val_f1 = f1_score(val_targets, (val_preds > 0.5).astype(int), average="macro")

    # ===== SAVE BEST MODEL =====
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "attention_fusion_model.pth"))

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

Epoch 1/25 | Loss: 415.8419 | Val F1: 0.5932
Epoch 2/25 | Loss: 156.5840 | Val F1: 0.6162
Epoch 3/25 | Loss: 86.2859 | Val F1: 0.8539
Epoch 4/25 | Loss: 71.3038 | Val F1: 0.8559
Epoch 5/25 | Loss: 65.9673 | Val F1: 0.8565
Epoch 6/25 | Loss: 63.3933 | Val F1: 0.8615
Epoch 7/25 | Loss: 61.7312 | Val F1: 0.8657
Epoch 8/25 | Loss: 60.7584 | Val F1: 0.8653
Epoch 9/25 | Loss: 59.9428 | Val F1: 0.8681
Epoch 10/25 | Loss: 59.3721 | Val F1: 0.8664
Epoch 11/25 | Loss: 58.9327 | Val F1: 0.8698
Epoch 12/25 | Loss: 58.5588 | Val F1: 0.8714
Epoch 13/25 | Loss: 58.2111 | Val F1: 0.8704
Epoch 14/25 | Loss: 57.9751 | Val F1: 0.8722
Epoch 15/25 | Loss: 57.8275 | Val F1: 0.8716
Epoch 16/25 | Loss: 57.4789 | Val F1: 0.8734
Epoch 17/25 | Loss: 57.2728 | Val F1: 0.8744
Epoch 18/25 | Loss: 57.0224 | Val F1: 0.8725
Epoch 19/25 | Loss: 56.9255 | Val F1: 0.8746
Epoch 20/25 | Loss: 56.7877 | Val F1: 0.8759
Epoch 21/25 | Loss: 56.5341 | Val F1: 0.8742
Epoch 22/25 | Loss: 56.3993 | Val F1: 0.8771
Epoch 23/25 | Los

In [ ]:
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "attention_fusion_model.pth")))
model.eval()

AttentionFusion(
  (attention): Sequential(
    (0): Linear(in_features=7, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=16, out_features=7, bias=True)
    (4): Sigmoid()
  )
  (classifier): Sequential(
    (0): Linear(in_features=7, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=4, bias=True)
  )
)

In [ ]:
thresholds = []

for i in range(4):
    best_t, best_f1 = 0.5, 0

    for t in np.linspace(0.65, 0.75, 200):
        preds_bin = (val_preds[:, i] > t).astype(int)
        f1 = f1_score(val_targets[:, i], preds_bin)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print("Optimal thresholds:", thresholds)

Optimal thresholds: [np.float64(0.6831658291457287), np.float64(0.6540201005025126), np.float64(0.65), np.float64(0.65)]


In [ ]:
thresholds_path = os.path.join(SAVE_DIR, "thresholds.npy")

np.save(thresholds_path, thresholds)

print("Thresholds saved at:", thresholds_path)
print("Thresholds:", thresholds)

Thresholds saved at: /content/drive/MyDrive/attention_fusion_results_final/thresholds.npy
Thresholds: [np.float64(0.6831658291457287), np.float64(0.6540201005025126), np.float64(0.65), np.float64(0.65)]


In [ ]:
model.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        out = model(xb)
        probs = torch.sigmoid(out / 1.2)

        test_preds.append(probs.cpu().numpy())
        test_targets.append(yb.cpu().numpy())

test_preds = np.vstack(test_preds)
test_targets = np.vstack(test_targets)

In [ ]:
final_preds = np.zeros_like(test_preds)

for i in range(4):
    final_preds[:, i] = (test_preds[:, i] > thresholds[i]).astype(int)

In [ ]:
preds_df = pd.DataFrame(test_preds, columns=["DR_prob", "Gl_prob", "AMD_prob", "DED_prob"])
labels_df = pd.DataFrame(final_preds, columns=["DR_pred", "Gl_pred", "AMD_pred", "DED_pred"])
gt_df = pd.DataFrame(test_targets, columns=["DR_gt", "Gl_gt", "AMD_gt", "DED_gt"])

full_df = pd.concat([preds_df, labels_df, gt_df], axis=1)

preds_path = os.path.join(SAVE_DIR, "test_predictions.csv")
full_df.to_csv(preds_path, index=False)

print("Predictions saved at:", preds_path)

Predictions saved at: /content/drive/MyDrive/attention_fusion_results_final/test_predictions.csv


In [ ]:
diseases = ["DR", "Glaucoma", "AMD", "DED"]

results = []

for i, disease in enumerate(diseases):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], final_preds[:, i])
    prec = precision_score(test_targets[:, i], final_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], final_preds[:, i])
    f1 = f1_score(test_targets[:, i], final_preds[:, i])

    results.append([disease, auc, acc, prec, rec, f1])

results_df = pd.DataFrame(results, columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"])
print(results_df)

    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.985400  0.965065   0.727612  0.698925  0.712980
1  Glaucoma  0.975913  0.937027   0.859396  0.787952  0.822124
2       AMD  0.999373  0.994882   0.995931  0.980962  0.988390
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
metrics_path = os.path.join(SAVE_DIR, "metrics.csv")

results_df.to_csv(metrics_path, index=False)

print("Metrics saved at:", metrics_path)

Metrics saved at: /content/drive/MyDrive/attention_fusion_results_final/metrics.csv


In [ ]:
# ===== MODEL-SPECIFIC THRESHOLD TUNING (ATTENTION FUSION) =====

thresholds = []

threshold_ranges = [
    np.linspace(0.6, 0.75, 200),   # DR
    np.linspace(0.55, 0.7, 200),   # Glaucoma
    np.linspace(0.1, 0.3, 100),    # AMD
    np.linspace(0.4, 0.6, 100)     # DED
]

for i in range(4):
    best_t, best_f1 = 0.5, 0

    for t in threshold_ranges[i]:
        preds_bin = (val_preds[:, i] > t).astype(int)
        f1 = f1_score(val_targets[:, i], preds_bin)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print("Optimized thresholds (Attention):", thresholds)

Optimized thresholds (Attention): [np.float64(0.6829145728643216), np.float64(0.599748743718593), np.float64(0.1), np.float64(0.5797979797979798)]


In [ ]:
# ===== OPTIMIZED THRESHOLD EVALUATION (ATTENTION) =====

opt_preds = np.zeros_like(test_preds)

for i in range(4):
    opt_preds[:, i] = (test_preds[:, i] > thresholds[i]).astype(int)

results_opt = []

for i, disease in enumerate(["DR", "Glaucoma", "AMD", "DED"]):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], opt_preds[:, i])
    prec = precision_score(test_targets[:, i], opt_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], opt_preds[:, i])
    f1 = f1_score(test_targets[:, i], opt_preds[:, i])

    results_opt.append([disease, auc, acc, prec, rec, f1])

results_opt_df = pd.DataFrame(
    results_opt,
    columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"]
)

print("=== Attention Fusion (Optimized Thresholds) ===")
print(results_opt_df)

=== Attention Fusion (Optimized Thresholds) ===
    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.985400  0.964619   0.722222  0.698925  0.710383
1  Glaucoma  0.975913  0.935692   0.839398  0.806024  0.822372
2       AMD  0.999373  0.995327   0.990955  0.987976  0.989463
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
# ===== SAVE ALL RESULTS (FIXED + OPTIMIZED) =====

# Save fixed threshold results
results_df.to_csv(
    os.path.join(SAVE_DIR, "metrics_fixed.csv"), index=False
)

# Save optimized threshold results
results_opt_df.to_csv(
    os.path.join(SAVE_DIR, "metrics_optimized.csv"), index=False
)

# Save thresholds
np.save(
    os.path.join(SAVE_DIR, "thresholds.npy"), thresholds
)

# (Optional) save predictions for analysis
preds_df = pd.DataFrame(test_preds, columns=["DR_prob", "Gl_prob", "AMD_prob", "DED_prob"])
opt_preds_df = pd.DataFrame(opt_preds, columns=["DR_pred", "Gl_pred", "AMD_pred", "DED_pred"])
gt_df = pd.DataFrame(test_targets, columns=["DR_gt", "Gl_gt", "AMD_gt", "DED_gt"])

full_df = pd.concat([preds_df, opt_preds_df, gt_df], axis=1)

full_df.to_csv(
    os.path.join(SAVE_DIR, "test_predictions.csv"), index=False
)

print("✅ All results saved successfully!")

✅ All results saved successfully!
